In [5]:

import numpy as np
import pandas as pd
import rioxarray as rxr
from osgeo import gdal
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline, make_pipeline

import geopandas as gpd

In [6]:
def save_img(output_filename, reference_filename, array):
    # 打开参考数据
    reference = gdal.Open(reference_filename)
    
    # 获取地理变换参数和坐标系
    geo_transform = reference.GetGeoTransform()
    projection = reference.GetProjection()

    # 获取矩阵尺寸
    rows, cols = array.shape

    # 创建新的 TIFF 数据集
    driver = gdal.GetDriverByName('GTiff')
    out_data = driver.Create(output_filename, cols, rows, 1, gdal.GDT_Float32)

    # 设置地理变换和坐标系
    out_data.SetGeoTransform(geo_transform)
    out_data.SetProjection(projection)
    

    # 写入数据
    out_data.GetRasterBand(1).WriteArray(array)

    # 保存并关闭文件
    out_data.FlushCache()
    out_data = None

In [25]:
import geopandas as gpd
import rasterio
from shapely.geometry import box
import pandas as pd
from shapely.geometry import Point
from rasterio.sample import sample_gen


location = 'Morecambe'
region = 'Region4'
input_bands = 'all_bands'
# input_bands = 'Freq'
method = 'RF'#,'RF','Poly, MpLinear', 'MpPoly'
year = 2019

csv_path = f'G:\ICESat-2\{region}\{region}_{location}\A_{location}_30m_{year}_new10.csv'
osgm_path = 'G:\OSGM15\Great_Britain_osgm15.tif'

raster_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\{location}_{year}_FB6_new5.tif'
output_raw_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\A_{location}_30m_{year}_raw_new10.shp'
output_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\A_{location}_30m_{year}_new10.shp'

#-------------------------------------------------------------------

# raster_path = 'G:\ICESat-2\Discussion5_1\Retrival_DTM\wash_bay_2022_FB6_new4.tif'
output_point = f'G:\ICESat-2\Discussion5_1\ICESAT\A_{location}_30m_{year}_new10.shp'
#-------------------------------------------------------------------

# raster_path = f'G:\ICESat-2\{region}_map\Dee_map\A{year}\Dee_{year}_FB6_new_.tif'
# output_raw_point = f'G:\ICESat-2\{region}_map\Dee_map\A{year}\A_Dee_30m_{year}_raw_new10.shp'
# output_point = f'G:\ICESat-2\{region}_map\Dee_map\A{year}\A_Dee_30m_{year}_new10.shp'

# path = f'F:\ICESat-2\Region4_map\{location}_map\A{year}\A_{location}_30m_{year}.shp'

df = pd.read_csv(csv_path)
lon, lat = 'mean_lon', 'mean_lat'
print(df.shape)
print(df.columns)

geometry = [Point(xy) for xy in zip(df[lon], df[lat])]
gdf = gpd.GeoDataFrame(df, geometry=geometry)
gdf.set_crs(epsg=4326, inplace=True)
# 读取点数据
points = gdf#//gpd.read_file(output_)

# load FB6
with rasterio.open(raster_path) as src:
    # 1. 检查坐标系
    if points.crs != src.crs:
        points = points.to_crs(src.crs)
    coords = [(point.x, point.y) for point in points.geometry]
    values = list(src.sample(coords))

    # 添加波段值到属性表
    band_count = src.count
    for band in range(band_count):
        points[f"b{band + 1}_FB6"] = [v[band] for v in values]
        
print('band_count', band_count)
#load osgm_path
with rasterio.open(osgm_path) as src:

    if points.crs != src.crs:
        points = points.to_crs(src.crs)
    coords = [(point.x, point.y) for point in points.geometry]
    values = list(src.sample(coords))

    band_count = src.count
    for band in range(band_count):
        points[f"b{band + 1}_osgb36"] = [v[band] for v in values]

    points['OH'] = points['mean_height'] - points['b1_osgb36'] 
    # 保存为新的 shapefile

#icesat_mask = (points['OH'] < 5) & (points['b7_FB6'] > 0)  
icesat_mask = (points['OH'] < 5) & (points['b10_FB6'] > 0) # & (points['b6_FB6'] < 0.95)
data = points[icesat_mask]
data.shape
print(points.columns)

# points.to_file(output_raw_point)
data.to_file(output_point)

(21403, 8)
Index(['Unnamed: 0', 'distance', 'mean_height', 'mean_lon', 'mean_lat', 'OSGM',
       'OH', 'Strip_No'],
      dtype='object')
band_count 18
Index(['Unnamed: 0', 'distance', 'mean_height', 'mean_lon', 'mean_lat', 'OSGM',
       'OH', 'Strip_No', 'geometry', 'b1_FB6', 'b2_FB6', 'b3_FB6', 'b4_FB6',
       'b5_FB6', 'b6_FB6', 'b7_FB6', 'b8_FB6', 'b9_FB6', 'b10_FB6', 'b11_FB6',
       'b12_FB6', 'b13_FB6', 'b14_FB6', 'b15_FB6', 'b16_FB6', 'b17_FB6',
       'b18_FB6', 'b1_osgb36'],
      dtype='object')


C:\Users\minyu\AppData\Local\Temp\ipykernel_51152\1766445434.py:82: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  data.to_file(output_point)
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed_ 0'
  ogr_write(
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'mean_height' to 'mean_heigh'
  ogr_write(


In [26]:
# target_band_all = ['b1_FB6','b3_FB6','b5_FB6','b8_FB6', 'b9_FB6', 'b4_FB6','b6_FB6','b7_FB6']#, 'b6_FB2', 'b7_FB2']
# target_id_all = [0, 2, 4, 7, 8, 3, 5, 6]
target_band_all = ['b1_FB6', 'b5_FB6', 'b14_FB6', 'b15_FB6']
target_id_all = [0, 4, 13, 14]
for bands in range(4,5):
    
    if input_bands == 'all_bands':

        # target_band = ['b1_FB6','b2_FB6','b3_FB6','b4_FB6']#,'b5_FB2', 'b6_FB2', 'b7_FB2']
        # target_id = [0, 1, 2, 3]#, 4, 5] #
        
        # target_band = ['b1_FB6','b3_FB6','b5_FB6','b8_FB6', 'b9_FB6', 'b4_FB6','b6_FB6','b7_FB6']#, 'b6_FB2', 'b7_FB2']
        # target_id = [0, 2, 4, 7, 8, 3, 5, 6]# 5] #
        target_band = target_band_all[:bands]
        target_id = target_id_all[:bands]
        # target_band = ['b1_FB6']#,'b5_FB2', 'b6_FB2', 'b7_FB2']
        # target_id = [0]#, 4, 5] #
        
        # target_band = ['b1_FB6',  'b2_FB6']#,'b5_FB2', 'b6_FB2', 'b7_FB2']
        # target_id = [0, 1]#, 4, 5] #

        #output_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\{location}_{year}_4_{method}_FB6_new10.tif'
        output_path = f'G:\ICESat-2\FB_importance_result\{location}_{year}_{bands}_{method}_FB17_new10.tif'
    else:
        target_band = ['b1_FB6'] # ['b1_ribble_']# ['b1_Morecam']#[f'b1_{location}_']4
        target_id = [0]    
        output_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\{location}_{year}_Freq_{method}_FB6_new10_A.tif'


    y_train = data['OH'].values
    x_train = data[target_band].values
    print('x_train:', x_train.shape)
    print('y_train', y_train.shape)

    #select calssifer
    if method == 'RF':
        #classifier = RandomForestRegressor(n_estimators=300, random_state=30)
        classifier = RandomForestRegressor(n_estimators=300, min_samples_leaf=30, random_state=30, max_depth = 12)
        #classifier = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=30)
        classifier.fit(x_train, y_train)
    elif method == 'Linear':
        classifier = np.polyfit(x_train[:, 0], y_train, deg=1)
    elif method == 'Poly':
        classifier = np.polyfit(x_train[:, 0], y_train, deg=2)
    elif method == 'MpLinear':
        classifier = LinearRegression()
        classifier.fit(x_train, y_train)
    elif method == 'MpPoly':
        degree = 3  # 设置多项式的阶数
        classifier = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        classifier.fit(x_train, y_train)
        # classifier = PolynomialFeatures()
        # classifier.fit(x_train, y_train)
    else:
        print('error classifier input')

    # # load Ramdon Forest classifer
    # rf_regressor = RandomForestRegressor(n_estimators=100, random_state=30)
    # rf_regressor.fit(x_train, y_train)

    #load raster 


    # raster_path = f'G:\ICESat-2\Discussion5_1\Retrival_DTM\wash_bay_2022_FB6_new2.tif'
    raster_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\{location}_{year}_FB6_new5.tif'

    raster = rxr.open_rasterio(raster_path).values
    raster = np.nan_to_num(raster, nan = 0)
    selected_raster = raster[target_id, :, :]

    if len(selected_raster.shape) != 3:
        row, col = selected_raster.shape
        selected_raster = selected_raster.reshape(1, row, col)
    else: 
        bands, row, col = selected_raster.shape

    print('selected_raster.shape', selected_raster.shape)
        
    # Topograph mapping 

    k = 0
    result = np.array([[]])
    for i in range(row):
        
        x_train = np.transpose(selected_raster[:, i, :])
        if method == 'RF':
            y_pred = classifier.predict(x_train)
        elif  method == 'Linear' :
            y_pred = np.polyval(classifier, x_train)
        elif  method == 'Poly' :
            y_pred = np.polyval(classifier, x_train)
        elif method == 'MpLinear':
            y_pred = classifier.predict(x_train)
        elif method == 'MpPoly':
            
            y_pred = classifier.predict(x_train)

        result = np.append(result, y_pred)
        if i%(int(row/5)) == 0:
            
            print(k*20 , '%_ is done')
            k = k+1

    print(result.shape, type(result))

    # tf_mask = raster[4,:,:].flatten()
    # Freq_s2 = raster[5,:,:].flatten()

    tf_mask = raster[16,:,:].flatten()
    Freq_s2 = raster[17,:,:].flatten()
    result[(tf_mask == 0) | (Freq_s2 > 0.95)] = 0
    result_ = result.reshape(row, col)

    save_img(output_path, raster_path, result_)
        

x_train: (14048, 4)
y_train (14048,)
selected_raster.shape (4, 2147, 1901)
0 %_ is done
20 %_ is done
40 %_ is done
60 %_ is done
80 %_ is done
100 %_ is done
(4081447,) <class 'numpy.ndarray'>


In [8]:
target_band = ['b1_FB6','b3_FB6','b5_FB6','b8_FB6', 'b9_FB6', 'b4_FB6','b6_FB6','b7_FB6']#, 'b6_FB2', 'b7_FB2']
target_id = [0, 2, 4, 7, 8, 3, 5, 6]
#extract y_train and x_train data
if input_bands == 'all_bands':
    bands = 4
    # target_band = ['b1_FB6','b2_FB6','b3_FB6','b4_FB6']#,'b5_FB2', 'b6_FB2', 'b7_FB2']
    # target_id = [0, 1, 2, 3]#, 4, 5] #
    
    # target_band = ['b1_FB6','b3_FB6','b5_FB6','b8_FB6', 'b9_FB6', 'b4_FB6','b6_FB6','b7_FB6']#, 'b6_FB2', 'b7_FB2']
    # target_id = [0, 2, 4, 7, 8, 3, 5, 6]# 5] #
    target_band = target_band[:bands]
    target_id = target_id[:bands]
    # target_band = ['b1_FB6']#,'b5_FB2', 'b6_FB2', 'b7_FB2']
    # target_id = [0]#, 4, 5] #
    
    # target_band = ['b1_FB6',  'b2_FB6']#,'b5_FB2', 'b6_FB2', 'b7_FB2']
    # target_id = [0, 1]#, 4, 5] #

    #output_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\{location}_{year}_4_{method}_FB6_new10.tif'
    output_path = f'G:\ICESat-2\FB_importance_result\{location}_{year}_{bands}_{method}_FB10_new10.tif'
else:
    target_band = ['b1_FB6'] # ['b1_ribble_']# ['b1_Morecam']#[f'b1_{location}_']4
    target_id = [0]    
    output_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\{location}_{year}_Freq_{method}_FB6_new10_A.tif'


y_train = data['OH'].values
x_train = data[target_band].values
print('x_train:', x_train.shape)
print('y_train', y_train.shape)

#select calssifer
if method == 'RF':
    #classifier = RandomForestRegressor(n_estimators=300, random_state=30)
    classifier = RandomForestRegressor(n_estimators=300, min_samples_leaf=30, random_state=30, max_depth = 12)
    #classifier = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=30)
    classifier.fit(x_train, y_train)
elif method == 'Linear':
    classifier = np.polyfit(x_train[:, 0], y_train, deg=1)
elif method == 'Poly':
    classifier = np.polyfit(x_train[:, 0], y_train, deg=2)
elif method == 'MpLinear':
    classifier = LinearRegression()
    classifier.fit(x_train, y_train)
elif method == 'MpPoly':
    degree = 3  # 设置多项式的阶数
    classifier = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    classifier.fit(x_train, y_train)
    # classifier = PolynomialFeatures()
    # classifier.fit(x_train, y_train)
else:
    print('error classifier input')

# # load Ramdon Forest classifer
# rf_regressor = RandomForestRegressor(n_estimators=100, random_state=30)
# rf_regressor.fit(x_train, y_train)

#load raster 


# raster_path = f'G:\ICESat-2\Discussion5_1\Retrival_DTM\wash_bay_2022_FB6_new2.tif'
raster_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\{location}_{year}_FB6_new4.tif'

raster = rxr.open_rasterio(raster_path).values
raster = np.nan_to_num(raster, nan = 0)
selected_raster = raster[target_id, :, :]

if len(selected_raster.shape) != 3:
    row, col = selected_raster.shape
    selected_raster = selected_raster.reshape(1, row, col)
else: 
    bands, row, col = selected_raster.shape

print('selected_raster.shape', selected_raster.shape)

x_train: (4160, 4)
y_train (4160,)
selected_raster.shape (4, 2375, 2894)


In [9]:
# Topograph mapping 

k = 0
result = np.array([[]])
for i in range(row):
    
    x_train = np.transpose(selected_raster[:, i, :])
    if method == 'RF':
        y_pred = classifier.predict(x_train)
    elif  method == 'Linear' :
        y_pred = np.polyval(classifier, x_train)
    elif  method == 'Poly' :
        y_pred = np.polyval(classifier, x_train)
    elif method == 'MpLinear':
        y_pred = classifier.predict(x_train)
    elif method == 'MpPoly':
        
        y_pred = classifier.predict(x_train)

    result = np.append(result, y_pred)
    if i%(int(row/5)) == 0:
        
        print(k*20 , '%_ is done')
        k = k+1

print(result.shape, type(result))

# tf_mask = raster[4,:,:].flatten()
# Freq_s2 = raster[5,:,:].flatten()

tf_mask = raster[9,:,:].flatten()
Freq_s2 = raster[10,:,:].flatten()
result[(tf_mask == 0) | (Freq_s2 > 0.95)] = 0
result_ = result.reshape(row, col)

save_img(output_path, raster_path, result_)

# tf_mask = raster[4,:,:].flatten()

# result[tf_mask == 0] = 0
# result_ = result.reshape(row, col)

# save_img(output_path, raster_path, result_)

0 %_ is done
20 %_ is done
40 %_ is done
60 %_ is done
80 %_ is done
(6873250,) <class 'numpy.ndarray'>


In [11]:
tf_mask = raster[9,:,:].flatten()
Freq_s2 = raster[10,:,:].flatten()
result[(tf_mask == 0) | (Freq_s2 > 0.95)] = 0
result[(tf_mask == 0)]# | (Freq_s2 > 0.95)] = 0
result_ = result.reshape(row, col)

save_img(output_path, raster_path, result_)

In [29]:
Wash_point_path = 'G:\ICESat-2\FB_importance\wash_bay_2022_sample_points_final.shp'
band_list2 = ["Freq", "mean_RE4", "mean_MNDWI","mean_NDWI","mean_NIR", "mean_RE1", "mean_RE2", "mean_RE3"]
df = gpd.read_file(Wash_point_path)
df_ = df[band_list2].values
df_v = df_#.reshape(16, -1)
df_v.shape

(6222, 8)

In [30]:
corr = pd.DataFrame(df_v).corr()
corr

,0,1,2,3,4,5,6,7
0,1.000000,0.908896,-0.910510,-0.982183,0.900390,0.762209,0.881191,0.890756
1,0.908896,1.000000,-0.873077,-0.903031,0.998120,0.935153,0.995041,0.997626
2,-0.910510,-0.873077,1.000000,0.901433,-0.858932,-0.707375,-0.838615,-0.848354
3,-0.982183,-0.903031,0.901433,1.000000,-0.895331,-0.756547,-0.876418,-0.887008
4,0.900390,0.998120,-0.858932,-0.895331,1.000000,0.945495,0.996979,0.998337
5,0.762209,0.935153,-0.707375,-0.756547,0.945495,1.000000,0.964104,0.955170
6,0.881191,0.995041,-0.838615,-0.876418,0.996979,0.964104,1.000000,0.999251
7,0.890756,0.997626,-0.848354,-0.887008,0.998337,0.955170,0.999251,1.000000
